# Differentiable & Probabilistic Programming

In [ ]:
# ! pip install diffrax daft numpyro

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import optax
import diffrax
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, SVI, Predictive, Trace_ELBO, autoguide

%matplotlib inline
%config InlineBackend.figure_format='retina'

jax.config.update("jax_enable_x64", True)

color_default = "firebrick"

# Matplotlib defaults
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": False,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 12,
    }
)

## 1. Differentiable Programming — Fitting an ODE

We start by showing how automatic differentiation lets us fit the parameters of a dynamical system directly. The key idea: if our simulator is written in a differentiable framework, we can compute $\nabla_\theta \mathcal{L}$ and use gradient-based optimization.

### 1.1 The damped harmonic oscillator

Consider the damped harmonic oscillator, governed by

$$\frac{d^2 x}{dt^2} + \gamma \frac{dx}{dt} + \omega_0^2 x = 0$$

where $\gamma$ is the damping coefficient and $\omega_0$ is the natural frequency. We rewrite this as a first-order system by introducing $v = dx/dt$:

$$\frac{d}{dt}\begin{pmatrix} x \\ v \end{pmatrix} = \begin{pmatrix} v \\ -\gamma v - \omega_0^2 x \end{pmatrix}$$

We'll implement this in JAX using `diffrax` — a differentiable ODE solver library built on JAX.

In [ ]:
def vector_field(t, y, args):
    """RHS of the damped oscillator ODE system.

    y = [x, v], args = (gamma, omega0)
    """
    x, v = y
    gamma, omega0 = args
    dxdt = v
    dvdt = -gamma * v - omega0**2 * x
    return jnp.array([dxdt, dvdt])


def simulate(theta, t_obs):
    """Solve the damped oscillator ODE and return x(t) at observation times.

    Args:
        theta: (gamma, omega0) parameters
        t_obs: 1-D array of observation times
    Returns:
        x(t_obs) — the position at each observation time
    """
    gamma, omega0 = theta
    term = diffrax.ODETerm(vector_field)
    solver = diffrax.Tsit5()
    y0 = jnp.array([1.0, 0.0])  # x(0) = 1, v(0) = 0
    saveat = diffrax.SaveAt(ts=t_obs)
    sol = diffrax.diffeqsolve(
        term,
        solver,
        t0=t_obs[0],
        t1=t_obs[-1],
        dt0=0.11,
        y0=y0,
        args=(gamma, omega0),
        saveat=saveat,
        max_steps=10000,
    )
    return sol.ys[:, 0]  # Return x(t) only

### 1.2 Exploring solutions

Let's see how different parameter choices affect the dynamics. We'll sweep over a few $(\gamma, \omega_0)$ values.

In [ ]:
t_fine = jnp.linspace(0.0, 10.0, 500)

param_sets = [
    (0.1, 2.0, "Underdamped (light)"),
    (0.3, 2.0, "Underdamped (moderate)"),
    (1.0, 2.0, "Underdamped (heavy)"),
    (4.0, 2.0, "Overdamped"),
]

colors = ["#1b9e77", "#d95f02", color_default, "#7570b3"]

fig, ax = plt.subplots(figsize=(8, 4))
for (gamma, omega0, label), c in zip(param_sets, colors):
    x_t = simulate((gamma, omega0), t_fine)
    ax.plot(t_fine, x_t, label=rf"$\gamma={gamma},\;\omega_0={omega0}$ — {label}", color=c, lw=2)

ax.set_xlabel("Time $t$")
ax.set_ylabel("Position $x(t)$")
ax.set_title("Damped harmonic oscillator — varying parameters")
ax.legend(fontsize=9)
ax.axhline(0, color="k", lw=0.5, ls="--")
plt.tight_layout()

### 1.3 Generate synthetic data

We pick "true" parameters $\gamma_\mathrm{true} = 0.3$, $\omega_{0,\mathrm{true}} = 2.0$, simulate the system, and add Gaussian noise to mimic real observations.

In [ ]:
# True parameters
gamma_true, omega0_true = 0.3, 2.0
sigma_noise = 0.05

# Observation times (sparser than the fine grid)
t_obs = jnp.linspace(0.0, 10.0, 60)

# Ground truth trajectory
x_true = simulate((gamma_true, omega0_true), t_obs)

# Noisy observations
key = jax.random.PRNGKey(42)
x_data = x_true + sigma_noise * jax.random.normal(key, shape=x_true.shape)

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
t_dense = jnp.linspace(0.0, 10.0, 500)
x_dense = simulate((gamma_true, omega0_true), t_dense)
ax.plot(t_dense, x_dense, color="k", lw=1.5, label="True trajectory")
ax.scatter(t_obs, x_data, s=15, color=color_default, zorder=5, label=f"Noisy data ($\\sigma={sigma_noise}$)")
ax.set_xlabel("Time $t$")
ax.set_ylabel("$x(t)$")
ax.set_title("Synthetic observations of a damped oscillator")
ax.legend()
plt.tight_layout()

### 1.4 Define the loss and compute gradients

Since our entire simulation pipeline — ODE solver included — is written in JAX, we can differentiate through it with `jax.grad`. The loss is the mean squared error between the simulated and observed trajectories.

In [ ]:
def loss_fn(theta, t_obs, x_data):
    """MSE between simulated and observed trajectory."""
    x_pred = simulate(theta, t_obs)
    return jnp.mean((x_pred - x_data) ** 2)


# Compute gradient at an initial guess
theta_init = (0.8, 3.0)  # Deliberately far from truth
loss_val = loss_fn(theta_init, t_obs, x_data)
grad_val = jax.grad(loss_fn)(theta_init, t_obs, x_data)

print(f"Initial guess:  gamma={theta_init[0]}, omega0={theta_init[1]}")
print(f"Loss at guess:  {loss_val:.6f}")
print(f"Gradient:       dL/dgamma={grad_val[0]:.6f}, dL/domega0={grad_val[1]:.6f}")

### 1.5 Fit via gradient descent with Adam

We use `optax.adam` to optimize $(\gamma, \omega_0)$. The entire optimization loop is differentiating *through the ODE solver* at every step.

In [ ]:
# Optimization with optax
lr = 3e-3
optimizer = optax.adam(lr)

# Initialize as JAX arrays so optax can track them
params = jnp.array([0.8, 3.0])  # (gamma, omega0) — initial guess
opt_state = optimizer.init(params)

n_steps = 500
loss_history = []
param_history = []

for i in range(n_steps):
    loss_val, grads = jax.value_and_grad(loss_fn)(tuple(params), t_obs, x_data)
    grads = jnp.array(grads)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    loss_history.append(float(loss_val))
    param_history.append(params.copy())

    if (i + 1) % 100 == 0:
        print(f"Step {i + 1:4d} | Loss: {loss_val:.6f} | gamma: {params[0]:.4f} | omega0: {params[1]:.4f}")

param_history = jnp.stack(param_history)
print(f"\nFinal: gamma={params[0]:.4f}, omega0={params[1]:.4f}")
print(f"True:  gamma={gamma_true}, omega0={omega0_true}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# (i) Loss curve
axes[0].semilogy(loss_history, color=color_default, lw=1.5)
axes[0].set_xlabel("Optimization step")
axes[0].set_ylabel("MSE loss")
axes[0].set_title("Loss curve")

# (ii) Parameter trajectories
axes[1].plot(param_history[:, 0], label=r"$\gamma$", color="#1b9e77", lw=1.5)
axes[1].axhline(gamma_true, color="#1b9e77", ls="--", lw=1, alpha=0.7)
axes[1].plot(param_history[:, 1], label=r"$\omega_0$", color="#d95f02", lw=1.5)
axes[1].axhline(omega0_true, color="#d95f02", ls="--", lw=1, alpha=0.7)
axes[1].set_xlabel("Optimization step")
axes[1].set_ylabel("Parameter value")
axes[1].set_title("Parameter convergence")
axes[1].legend()

# (iii) Fitted trajectory vs data vs truth
x_fit = simulate(tuple(params), t_dense)
axes[2].plot(t_dense, x_dense, "k-", lw=1.5, label="Truth")
axes[2].scatter(t_obs, x_data, s=15, color=color_default, zorder=5, label="Data")
axes[2].plot(t_dense, x_fit, "--", color="#1b9e77", lw=2, label="Fitted")
axes[2].set_xlabel("Time $t$")
axes[2].set_ylabel("$x(t)$")
axes[2].set_title("Fitted trajectory")
axes[2].legend(fontsize=9)

plt.tight_layout()

## 2. Probabilistic Programming — Getting a Posterior

Gradient descent gives us a point estimate. But in science we want **uncertainty**: how confident are we in these parameter values? What range of trajectories is consistent with the data?

Let's express the same model as a probabilistic program using NumPyro and obtain a full posterior distribution $p(\gamma, \omega_0, \sigma \mid \mathbf{x}_\mathrm{obs})$.

### 2.1 Define the NumPyro model

We place priors on the physical parameters $\gamma$, $\omega_0$ and the noise scale $\sigma$, then use our differentiable `simulate` function inside the likelihood. This is a *differentiable probabilistic program*: the ODE solver runs inside the model.

In [ ]:
def oscillator_model(t_obs, x_obs=None):
    """NumPyro probabilistic model for the damped oscillator."""

    # Priors
    gamma = numpyro.sample("gamma", dist.Uniform(0.01, 2.0))
    omega0 = numpyro.sample("omega0", dist.Uniform(0.5, 5.0))
    sigma = numpyro.sample("sigma", dist.HalfNormal(0.5))

    # Forward model (differentiable ODE solve)
    x_pred = numpyro.deterministic("x_pred", simulate((gamma, omega0), t_obs))

    # Likelihood
    with numpyro.plate("obs", len(t_obs)):
        numpyro.sample("x_obs", dist.Normal(x_pred, sigma), obs=x_obs)

### 2.2 Run NUTS

Since our model is fully differentiable, we can use the No-U-Turn Sampler (NUTS) — an efficient variant of Hamiltonian Monte Carlo that exploits gradient information to explore the posterior.

In [ ]:
nuts_kernel = NUTS(oscillator_model)
mcmc = MCMC(nuts_kernel, num_warmup=100, num_samples=10000)
mcmc.run(jax.random.PRNGKey(0), t_obs, x_obs=x_data)
mcmc.print_summary()

In [ ]:
samples = mcmc.get_samples()

### 2.3 Posterior corner plot

Let's visualize the joint posterior on $(\gamma, \omega_0, \sigma)$ and mark the true values.

In [ ]:
import corner

posterior_array = np.array(
    [
        samples["gamma"],
        samples["omega0"],
        samples["sigma"],
    ]
).T

fig = corner.corner(
    posterior_array,
    labels=[r"$\gamma$", r"$\omega_0$", r"$\sigma$"],
    truths=[gamma_true, omega0_true, sigma_noise],
    truth_color=color_default,
    show_titles=True,
    title_fmt=".3f",
)
fig.suptitle("Posterior from NUTS", y=1.02, fontsize=14)

### 2.4 Posterior predictive

Having obtained posterior samples, we can propagate them through the simulator to get a *distribution over trajectories*. This is the posterior predictive: it shows the range of dynamical behaviors consistent with the data.

In [ ]:
# Draw 100 posterior trajectories
n_draw = 100
idx = jax.random.choice(jax.random.PRNGKey(1), len(samples["gamma"]), shape=(n_draw,), replace=False)

fig, ax = plt.subplots(figsize=(8, 4))

x_post = jax.vmap(lambda g, w: simulate((g, w), t_dense))(samples["gamma"][idx], samples["omega0"][idx])
for i in range(n_draw):
    ax.plot(t_dense, x_post[i], color=color_default, alpha=0.08, lw=0.8)

ax.plot(t_dense, x_dense, "k-", lw=1.5, label="Truth")
ax.scatter(t_obs, x_data, s=15, color="k", zorder=5, label="Data")

ax.set_xlabel("Time $t$")
ax.set_ylabel("$x(t)$")
ax.set_title("Posterior predictive — 100 draws")
ax.legend()
plt.tight_layout()

## 3. Hierarchical Model — Supernova Cosmology

Now let's apply the same differentiable probabilistic programming approach to a real physics problem: inferring cosmological parameters from Type Ia supernova observations.

### 3.1 The physics

Type Ia supernovae are *standard candles*: their intrinsic luminosity is approximately constant. The observed brightness depends on the luminosity distance $d_L$, which in turn encodes the expansion history of the Universe.

**Distance modulus:**
$$\mu(z) = 25 + 5\log_{10}\!\left(\frac{d_L}{\mathrm{Mpc}}\right)$$

**Luminosity distance** (flat Universe):
$$d_L(z) = (1 + z)\int_0^z \frac{c}{H(z')}\,dz'$$

**Hubble parameter** (matter + cosmological constant):
$$H(z) = H_0\sqrt{\Omega_m(1+z)^3 + (1 - \Omega_m)}$$

Our goal: infer $H_0$ and $\Omega_m$ from measurements of $\mu(z)$.

### 3.2 Implement the forward model

We compute the luminosity distance by numerical integration using `jnp.trapezoid`, then vectorize over redshifts with `jax.vmap`.

In [ ]:
c_light = 3e5  # km/s


def _hubble(z, H0, Om):
    """Hubble parameter H(z) for flat LCDM."""
    return H0 * jnp.sqrt(Om * (1 + z) ** 3 + (1 - Om))


def _distance_modulus_single(z, H0, Om):
    """Distance modulus for a single redshift value."""
    z_grid = jnp.linspace(0.0, z, 100)
    integrand = c_light / _hubble(z_grid, H0, Om)
    dL = (1 + z) * jnp.trapezoid(integrand, z_grid)
    return 25.0 + 5.0 * jnp.log10(dL)


# Vectorize over the redshift axis and JIT compile
distance_modulus = jax.jit(jax.vmap(_distance_modulus_single, in_axes=(0, None, None)))

In [ ]:
# Quick sanity check
z_test = jnp.linspace(0.01, 1.5, 100)
mu_test = distance_modulus(z_test, 70.0, 0.3)
print(f"mu(z=0.5, H0=70, Om=0.3) = {distance_modulus(jnp.array([0.5]), 70.0, 0.3)[0]:.2f}")

### 3.3 The dataset: SCP Union2.1

We use the [Supernova Cosmology Project "Union" compilation](https://supernova.lbl.gov/Union/) of Type Ia supernova distance moduli — 580 supernovae spanning redshifts $z \sim 0.01$ to $\sim 1.4$.

Each data point gives us a redshift $z_n$, a measured distance modulus $\mu_n^\mathrm{obs}$, and its measurement uncertainty $\sigma_n$.

In [ ]:
import os
import urllib.request

# Download SCP Union2.1 supernova data
url = "https://supernova.lbl.gov/union/figures/SCPUnion2.1_mu_vs_z.txt"
os.makedirs("./data", exist_ok=True)
data_path = "./data/SCPUnion2.1_mu_vs_z.txt"

if not os.path.exists(data_path):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response, open(data_path, "wb") as f:
        f.write(response.read())
    print(f"Downloaded to {data_path}")
else:
    print(f"Already cached at {data_path}")

# Load: columns are [index, redshift, distance_modulus, error]
data = np.genfromtxt(data_path)
z_sn = jnp.array(data[:, 1])  # Redshift
mu_obs = jnp.array(data[:, 2])  # Distance modulus
mu_err = jnp.array(data[:, 3])  # Measurement uncertainty

print(f"Loaded {len(z_sn)} supernovae, z range: [{z_sn.min():.3f}, {z_sn.max():.3f}]")

# Hubble diagram
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(
    np.array(z_sn),
    np.array(mu_obs),
    yerr=np.array(mu_err),
    fmt=".",
    color="k",
    ms=4,
    elinewidth=0.6,
    alpha=0.5,
    label="SCP Union2.1",
)
z_smooth = jnp.linspace(0.01, 1.5, 200)
ax.plot(z_smooth, distance_modulus(z_smooth, 70.0, 0.3), color=color_default, lw=1.5, label=r"$H_0=70,\;\Omega_m=0.3$")
ax.set_xlabel("Redshift $z$")
ax.set_ylabel("Distance modulus $\\mu$")
ax.set_title("SCP Union2.1 — 580 Type Ia supernovae")
ax.legend()
plt.tight_layout()

### 3.4 Define the NumPyro model

We define a hierarchical model: broad priors on $H_0$ and $\Omega_m$, with each supernova observation conditionally independent given the cosmological parameters.

In [ ]:
def cosmo_model(z_obs, mu_err_obs, mu_obs=None):
    """NumPyro model for supernova cosmology."""

    # Priors on cosmological parameters
    H0 = numpyro.sample("H0", dist.Uniform(50.0, 100.0))
    Om = numpyro.sample("Om", dist.Uniform(0.0, 1.0))

    # Predicted distance moduli (differentiable forward model)
    mu_pred = numpyro.deterministic("mu_pred", distance_modulus(z_obs, H0, Om))

    # Likelihood: each SN is an independent Gaussian observation
    with numpyro.plate("sn", len(z_obs)):
        numpyro.sample("mu_obs", dist.Normal(mu_pred, mu_err_obs), obs=mu_obs)

### 3.5 Inference with HMC (NUTS)

Let's run NUTS on the cosmological model. The classic banana-shaped degeneracy between $H_0$ and $\Omega_m$ should be visible in the posterior.

In [ ]:
nuts_cosmo = NUTS(cosmo_model)
mcmc_cosmo = MCMC(nuts_cosmo, num_warmup=500, num_samples=2000)
mcmc_cosmo.run(jax.random.PRNGKey(42), z_sn, mu_err, mu_obs=mu_obs)
mcmc_cosmo.print_summary(exclude_deterministic=True)

In [ ]:
cosmo_samples = mcmc_cosmo.get_samples()

cosmo_array_hmc = np.array([cosmo_samples["H0"], cosmo_samples["Om"]]).T

fig = corner.corner(
    cosmo_array_hmc,
    labels=[r"$H_0$", r"$\Omega_m$"],
    show_titles=True,
    title_fmt=".2f",
)
fig.suptitle("HMC posterior — supernova cosmology", y=1.02, fontsize=14)

### 3.6 Inference with Stochastic Variational Inference (SVI)

As an alternative to MCMC, we can use variational inference to approximate the posterior with a parametric family. We use `AutoMultivariateNormal` — a multivariate Gaussian that captures correlations between $H_0$ and $\Omega_m$.

In [ ]:
# Define guide and SVI
guide_cosmo = autoguide.AutoMultivariateNormal(cosmo_model)
svi_optimizer = numpyro.optim.Adam(1e-3)
svi_cosmo = SVI(cosmo_model, guide_cosmo, svi_optimizer, Trace_ELBO())

# Run SVI
svi_result = svi_cosmo.run(jax.random.PRNGKey(0), 5000, z_sn, mu_err, mu_obs=mu_obs)

In [ ]:
# Draw samples from the SVI guide and compare with HMC
svi_params = svi_result.params
svi_samples = guide_cosmo.sample_posterior(jax.random.PRNGKey(1), svi_params, sample_shape=(5000,))
cosmo_array_svi = np.array([svi_samples["H0"], svi_samples["Om"]]).T

# Corner plot comparison
import matplotlib.lines

fig = corner.corner(
    cosmo_array_hmc,
    labels=[r"$H_0$", r"$\Omega_m$"],
    color="k",
)
corner.corner(
    cosmo_array_svi,
    fig=fig,
    color="#1b9e77",
    weights=np.ones(len(cosmo_array_svi)) / (len(cosmo_array_svi) / len(cosmo_array_hmc)),
)

black_line = matplotlib.lines.Line2D([], [], color="k", label="HMC")
green_line = matplotlib.lines.Line2D([], [], color="#1b9e77", label="SVI")
plt.legend(handles=[black_line, green_line], bbox_to_anchor=(0.0, 1.0, 1.0, 0.0), loc=4)
fig.suptitle("HMC vs SVI posterior comparison", y=1.02, fontsize=14)

The Gaussian variational family (SVI) may struggle to capture the banana-shaped degeneracy that HMC can sample exactly. This is a key trade-off: SVI is faster but makes approximations; HMC gives exact samples but can be slower.

### 3.7 Posterior predictive — Hubble diagram

Finally, let's propagate our posterior uncertainty onto the Hubble diagram by drawing distance modulus curves from the HMC posterior and plotting the 95% credible band.

In [ ]:
# Compute posterior predictive distance modulus curves
z_plot = jnp.linspace(0.01, 1.5, 200)
mu_post_curves = jnp.array(
    [
        distance_modulus(z_plot, float(cosmo_samples["H0"][i]), float(cosmo_samples["Om"][i]))
        for i in range(len(cosmo_samples["H0"]))
    ]
)

# Percentiles
mu_median = np.percentile(mu_post_curves, 50, axis=0)
mu_lo = np.percentile(mu_post_curves, 2.5, axis=0)
mu_hi = np.percentile(mu_post_curves, 97.5, axis=0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(z_plot, mu_lo, mu_hi, alpha=0.3, color=color_default, label="95% credible band (HMC)")
ax.plot(z_plot, mu_median, color=color_default, lw=1.5, label="Posterior median")
ax.errorbar(
    np.array(z_sn),
    np.array(mu_obs),
    yerr=np.array(mu_err),
    fmt=".",
    color="k",
    ms=4,
    elinewidth=0.6,
    alpha=0.5,
    label="SCP Union2.1",
)

ax.set_xlabel("Redshift $z$")
ax.set_ylabel("Distance modulus $\\mu$")
ax.set_title("Posterior predictive Hubble diagram")
ax.legend(fontsize=9)
plt.tight_layout()